<a href="https://colab.research.google.com/github/hussainiram0601-lang/KerasTuner-DNN-Optimized/blob/main/KerasTuner_DNN_Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [38]:
df = pd.read_csv('/content/diabetes.csv')

In [39]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [40]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [41]:
X = df.iloc[:,:-1].values

In [42]:
y = df.iloc[:,-1].values


In [43]:
from sklearn.preprocessing import StandardScaler
sc= StandardScaler()
X = sc.fit_transform(X)

In [44]:
from sklearn.model_selection import train_test_split
X_train  , X_test, y_train , y_test = train_test_split(X, y , test_size = 0.2 , random_state=42)

In [55]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout


In [46]:
from keras.src.metrics.accuracy_metrics import accuracy
model = Sequential()
model.add(Dense(32 , activation='relu' , input_dim = 8))
model.add(Dense(16 , activation ='relu'))
model.add(Dense(1 , activation='sigmoid'))

model.compile(optimizer = 'Adam' , loss = 'binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [47]:
model.fit(X_train, y_train, batch_size = 32, epochs = 100 , validation_data=(X_test, y_test) )

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6645 - loss: 0.6491 - val_accuracy: 0.6818 - val_loss: 0.6484
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7427 - loss: 0.5752 - val_accuracy: 0.6948 - val_loss: 0.6184
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7508 - loss: 0.5418 - val_accuracy: 0.7013 - val_loss: 0.6022
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7427 - loss: 0.5168 - val_accuracy: 0.7143 - val_loss: 0.5899
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7541 - loss: 0.4988 - val_accuracy: 0.7208 - val_loss: 0.5790
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7573 - loss: 0.4871 - val_accuracy: 0.7208 - val_loss: 0.5747
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7655 - loss: 0.4766 - val_accuracy: 0.7273 - val_loss: 0.5692
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7687 - loss: 0.4695 - val_accuracy: 0.720

In [48]:
pip install -U keras-tuner

In [49]:
import kerastuner as kt

In [61]:
from keras.src.optimizers import adadelta
def build_model(hp):
  model = Sequential()
  cnt = 0
  for i in range(hp.Int('num_layers', min_value=1 , max_value = 10)):
    if cnt == 0:
      model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value = 128),
                      activation= hp.Choice('activation'+str(i),values=['relu','sigmoid','tanh','softmax']),
                      input_dim=8))
      model.add(Dropout(hp.Choice('dropout'+str(i),values =[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
      model.add(Dense(hp.Int('units'+str(i),min_value=8,max_value = 128),
                      activation= hp.Choice('activation'+str(i),values=['relu','sigmoid','tanh','softmax']),
                      ))
      model.add(Dropout(hp.Choice('dropout'+str(i),values =[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    cnt+=1
  model.add(Dense(1, activation= 'sigmoid'))
  model.compile(optimizer = hp.Choice('optimizer',values=['adam','adagrad','adadelta','rmsprop','sgd','nadam']),
                loss = 'binary_crossentropy',
                metrics = ['accuracy'])
  return model



In [66]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy',max_trials=3,directory='mydir',project_name = 'Final2')
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 06s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6688311696052551
Total elapsed time: 00h 00m 16s


In [63]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 60,
 'activation0': 'relu',
 'optimizer': 'rmsprop',
 'units1': 50,
 'activation1': 'sigmoid',
 'units2': 43,
 'activation2': 'tanh',
 'units3': 73,
 'activation3': 'sigmoid',
 'units4': 49,
 'activation4': 'relu',
 'units5': 57,
 'activation5': 'tanh',
 'units6': 12,
 'activation6': 'sigmoid',
 'units7': 86,
 'activation7': 'sigmoid',
 'units8': 37,
 'activation8': 'tanh'}

In [67]:
model = tuner.get_best_models(num_models=1)[0]

In [69]:
model.fit(X_train, y_train, epochs= 200 , initial_epoch=6, validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7345 - loss: 0.5548 - val_accuracy: 0.7792 - val_loss: 0.5307
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7492 - loss: 0.5253 - val_accuracy: 0.7727 - val_loss: 0.5301
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7182 - loss: 0.5482 - val_accuracy: 0.7792 - val_loss: 0.5288
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7150 - loss: 0.5579 - val_accuracy: 0.7662 - val_loss: 0.5338
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7313 - loss: 0.5361 - val_accuracy: 0.7792 - val_loss: 0.5293
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7410 - loss: 0.5207 - val_accuracy: 0.7792 - val_loss: 0.5284
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7313 - loss: 0.5342 - val_accuracy: 0.7792 - val_loss: 0.5278
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7443 - loss: 0.5153 - val_accuracy: 0.7